In [ ]:
import sys, os, re
sys.path.insert(0, '../..')
sys.path.insert(0, os.path.join('../..', 'utils'))
sys.path.insert(0, os.path.join('../..', 'utils', 'model_training'))

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from pathlib import Path
import math

import config
from pipeline_utils import get_exp_paths
from model_utils import (
    build_neighbor_curve_stack,
    _QuerySlice, _AttnScores, _WeightedRecon,
)
# model_utils_mtl defines its own (structurally identical) copies of these layers.
# SupCon1/2/3 attn-recon models are built via model_utils_mtl's embedding builder,
# so their saved _WeightedRecon layer is an instance of *this* class, not model_utils's.
from model_utils_mtl import (
    _QuerySlice as _QuerySliceMTL,
    _AttnScores as _AttnScoresMTL,
    _WeightedRecon as _WeightedReconMTL,
)

In [ ]:
EXP_FOLDER     = config.FINAL_EXP_FOLDER
GROUP_NAME     = "final_6_new"
OUTLIER_FILTER = None
CURVE_TYPE     = "ori_curve_sg_p4_norm"   # ori_curve | ori_curve_avg | ori_curve_wavelet_sym8
N_SAMPLE       = 9            # curves shown per well in neighbour grid
K_NEIGHBORS    = 24
SUPCON         = 0             # 0=base | 1=supcon | 3=supcon3


In [ ]:
# ── Data loading ──────────────────────────────────────────────────────────────

def load_experiment_data(base_datasets, exp_name, exp_folder_idx=None, curve_type=None,
                         outlier_filter=None, exp_path=None):
    """Returns X_AC (N,T), coords (N,2), well_ids (N,), Y_well (N,), X_time (T,), exp_path.
    Pass either exp_folder_idx (index into get_exp_paths(base_datasets/exp_name)) or an
    already-resolved exp_path directly (e.g. when looping a CROSS_DATASET_GROUPS list)."""
    if exp_path is None:
        exp_paths = get_exp_paths(os.path.join(base_datasets, exp_name))
        exp_path  = exp_paths[exp_folder_idx]
    print(f"Experiment: {exp_path.name}")

    td            = joblib.load(exp_path / config.TRAINING_DATA_PATH)
    dataset_names = td["dataset_name"]
    Y_well        = np.array(td["Y_well"])

    actual_name = config.CURVE_TYPE_ALIASES.get(curve_type, curve_type)
    if actual_name not in dataset_names:
        raise ValueError(f"curve_type '{curve_type}' not in dataset. Available: {dataset_names}")
    X_AC = np.array(td["dataset"][dataset_names.index(actual_name)], dtype=np.float32)
    X_time = np.asarray(td["timestamps"], dtype=float)

    metadata_df = pd.DataFrame(td["metadata"])
    coords = np.stack([
        metadata_df["pixel_row_idx"].values.astype(float),
        metadata_df["pixel_col_idx"].values.astype(float),
    ], axis=1)
    well_ids = (metadata_df["well_id"].values if "well_id" in metadata_df.columns
                else Y_well.copy())

    if outlier_filter is not None:
        filter_path = exp_path / f"{outlier_filter}_outlier" / "outlier_labels.joblib"
        if not filter_path.exists():
            raise FileNotFoundError(f"Outlier mask not found: {filter_path}")
        mask = joblib.load(filter_path) == 0
        X_AC, coords, well_ids, Y_well = X_AC[mask], coords[mask], well_ids[mask], Y_well[mask]
        print(f"  Filter '{outlier_filter}': kept {mask.sum()} / {len(mask)} samples")

    print(f"  X_AC: {X_AC.shape}")
    return X_AC, coords, well_ids, Y_well, X_time, exp_path


# ── Reconstruction ────────────────────────────────────────────────────────────

def build_recon_attn(X_AC, coords, well_ids, model_path, k=24):
    """Returns (neighbor_stack, X_recon, attn_weights (N, k+1))."""
    import tensorflow as tf

    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(f"No saved model at {model_path}")

    model = tf.keras.models.load_model(
        str(model_path),
        custom_objects={"_QuerySlice": _QuerySlice,
                        "_AttnScores": _AttnScores,
                        "_WeightedRecon": _WeightedRecon,
                        "_QuerySliceMTL": _QuerySliceMTL,
                        "_AttnScoresMTL": _AttnScoresMTL,
                        "_WeightedReconMTL": _WeightedReconMTL},
    )
    print(f"  Loaded: {model_path.name}  input={model.input_shape}")

    input_shape = model.input_shape
    if isinstance(input_shape, list):
        input_shape = next((s for s in input_shape if s[1] is not None), input_shape[0])
    if input_shape[1] is not None and input_shape[1] - 1 != k:
        print(f"  [!] Model expects k={input_shape[1] - 1} neighbours, not the requested k={k} -- using k={input_shape[1] - 1}.")
        k = input_shape[1] - 1

    stack        = build_neighbor_curve_stack(X_AC, coords, well_ids, k=k)
    # SC0 attn-recon models are built with model_utils's own _WeightedRecon; SupCon1/2/3
    # variants go through model_utils_mtl's embedding builder instead, so their saved
    # layer is model_utils_mtl's (distinct, if structurally identical) class -- match either.
    recon_layer  = next(l for l in model.layers if isinstance(l, (_WeightedRecon, _WeightedReconMTL)))
    recon_model  = tf.keras.Model(model.input, recon_layer.output)
    X_recon      = recon_model.predict(stack, verbose=0)[:, 0, :]
    attn_layer   = model.get_layer("attn_weights")
    attn_model   = tf.keras.Model(model.input, attn_layer.output)
    attn_weights = attn_model.predict(stack, verbose=0)[:, 0, :]

    return stack, X_recon, attn_weights


def _attn_model_path(exp_path, model_name, outlier_filter, curve_type):
    tag = str(outlier_filter) if outlier_filter is not None else "None"
    return exp_path / "ablations" / "model_interpretation" / f"{model_name}_{tag}_{curve_type}_model.keras"


# ── Plots ─────────────────────────────────────────────────────────────────────

_MAX_GREY = 100  # max individual curves drawn per panel; rest subsampled (memory / storage cap)

_CATEGORICAL_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                        "#e87ba4", "#008300", "#4a3aa7", "#e34948"]  # fixed order, CVD-checked


def plot_well_comparison(X_AC, X_attn, well_ids, exp_path, supcon, attn_model, rng_seed=42):
    """3-column per-well figure: X_AC | attn | mean overlay.

    Each row = one well. Up to _MAX_GREY individual grey curves shown (rasterised).
    """
    rng          = np.random.default_rng(rng_seed)
    unique_wells = np.unique(well_ids)
    n_wells      = len(unique_wells)
    t            = np.arange(X_AC.shape[1])

    # Resolve human-readable well labels from config
    well_label_map = config.get_label_mappings(exp_path).get(exp_path.name, {})

    col_labels  = ["Original (X_AC)", f"Attn recon\n{attn_model}", "Mean overlay"]
    bold_colors = ["steelblue", "seagreen"]

    fig, axes = plt.subplots(n_wells, 3,
                             figsize=(3 * 4.0, n_wells * 2.6),
                             facecolor="white", sharey="row")
    if n_wells == 1:
        axes = axes[np.newaxis, :]

    for row_i, well in enumerate(unique_wells):
        mask      = well_ids == well
        n_in_well = int(mask.sum())
        groups    = [X_AC[mask], X_attn[mask] if X_attn is not None else None]
        means     = [g.mean(axis=0) if g is not None else None for g in groups]

        well_label = well_label_map.get(int(well), str(well))

        # Subsample indices for grey background curves
        sub_idx = (rng.choice(n_in_well, size=min(_MAX_GREY, n_in_well), replace=False)
                   if n_in_well > _MAX_GREY else np.arange(n_in_well))

        for col_i, (curves, mean, color) in enumerate(zip(groups, means, bold_colors)):
            ax = axes[row_i, col_i]
            if curves is None:
                ax.text(0.5, 0.5, "model not saved",
                        transform=ax.transAxes, ha="center", va="center",
                        fontsize=8, color="grey")
                ax.set_axis_off()
            else:
                for ci in sub_idx:
                    ax.plot(t, curves[ci], color="grey", lw=0.5, alpha=0.15,
                            rasterized=True)
                ax.plot(t, mean, color=color, lw=2.0)
                ax.tick_params(labelsize=6)
                ax.grid(True, color="grey", alpha=0.15, linestyle=":")

                if col_i == 0:
                    ax.text(0.01, 0.97, f"well {well} — {well_label}  (n={n_in_well})",
                            transform=ax.transAxes, va="top", ha="left",
                            fontsize=8, fontweight="bold", color=color,
                            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, lw=0))

            if row_i == 0:
                ax.set_title(col_labels[col_i], fontsize=9, fontweight="bold", pad=6)

        # Mean overlay column
        ax_ov = axes[row_i, 2]
        ax_ov.plot(t, means[0], color="steelblue", lw=1.8, label="X_AC")
        if means[1] is not None:
            ax_ov.plot(t, means[1], color="seagreen", lw=1.8, label="attn")
        ax_ov.text(0.01, 0.97, f"well {well} — {well_label}",
                   transform=ax_ov.transAxes, va="top", ha="left",
                   fontsize=8, fontweight="bold", color="steelblue",
                   bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, lw=0))
        if row_i == 0:
            ax_ov.set_title(col_labels[2], fontsize=9, fontweight="bold", pad=6)
            ax_ov.legend(fontsize=7, framealpha=0.8)
        ax_ov.tick_params(labelsize=6)
        ax_ov.grid(True, color="grey", alpha=0.15, linestyle=":")

    supcon_label = f"SupCon v{supcon}" if supcon > 0 else "Base"
    fig.suptitle(f"Recon comparison — {supcon_label}  |  {exp_path.name}",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    return fig


def plot_recon_grid_combined(neighbor_stack, X_attn, indices, attn_model, exp_path,
                             attn_weights=None):
    """Per-sample grid: 1 row per sample, col 1 = own + neighbours, col 2 = attn recon."""
    n_rows = len(indices)
    fig, axes = plt.subplots(n_rows, 2, figsize=(12, n_rows * 4.4),
                             facecolor="white", sharey="row")
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    t = np.arange(neighbor_stack.shape[2])

    for row_i, idx in enumerate(indices):
        ax_own, ax_attn = axes[row_i, 0], axes[row_i, 1]

        for k_i in range(1, neighbor_stack.shape[1]):
            alpha = np.clip(float(attn_weights[idx, k_i]) * 5, 0.08, 0.9) if attn_weights is not None else 0.25
            ax_own.plot(t, neighbor_stack[idx, k_i], color="steelblue", lw=1.4, alpha=alpha,
                        linestyle="--", rasterized=True)
        ax_own.plot(t, neighbor_stack[idx, 0], color="orange", lw=2.2, label="main_pixels")
        ax_own.set_title(f"#{idx}  own + neighbours", fontsize=8)
        ax_own.tick_params(labelsize=6)
        ax_own.grid(True, color="grey", alpha=0.2, linestyle=":")
        if row_i == 0:
            ax_own.legend(fontsize=7, framealpha=0.8)

        if X_attn is not None:
            ax_attn.plot(t, X_attn[idx], color="seagreen", lw=2.2, label="attn_recon")
        ax_attn.set_title(f"#{idx}  attn recon", fontsize=8)
        ax_attn.tick_params(labelsize=6)
        ax_attn.grid(True, color="grey", alpha=0.2, linestyle=":")
        if row_i == 0:
            ax_attn.legend(fontsize=7, framealpha=0.8)

    fig.suptitle(
        f"Neighbour grid — attn={attn_model}\n{exp_path.name}",
        fontsize=10, fontweight="bold", y=1.005,
    )
    plt.tight_layout()
    return fig


def plot_all_curves_by_label(X, well_ids, exp_path, X_name="X",
                             max_per_well=None, alpha=0.05, rng_seed=42,
                             figsize=(11, 5.5), y_limit=None, with_nc=True, with_pc=True, nc_only=False,
                             ax=None, caption=None, x_time=None, show_legend=True, legend_title="label"):
    fig = None
    if ax is None:
        if X is None:
            raise ValueError(f"{X_name} is None (model not saved?) — nothing to plot")
        fig, ax = plt.subplots(figsize=figsize, facecolor="white")

    if X is None:
        ax.text(0.5, 0.5, "model not saved", transform=ax.transAxes,
                ha="center", va="center", fontsize=10, color="grey")
        ax.set_axis_off()
        return fig

    rng          = np.random.default_rng(rng_seed)
    unique_wells = np.unique(well_ids)
    t            = np.asarray(x_time) if x_time is not None else np.arange(X.shape[1])

    well_label_map = config.get_label_mappings(exp_path).get(exp_path.name, {})
    well_labels     = [well_label_map.get(int(w), str(w)) for w in unique_wells]
    unique_labels   = list(dict.fromkeys(well_labels))
    label_color     = {lab: _CATEGORICAL_PALETTE[i % len(_CATEGORICAL_PALETTE)]
                       for i, lab in enumerate(unique_labels)}

    for well, label in zip(unique_wells, well_labels):  # background: every curve, coloured by label, low alpha
        if(with_nc==False and label=="NC-ALL"):
            continue
        if(with_pc==False and label=="PC"):
            continue
        if(nc_only==True and label!="NC-ALL"):
            continue
        mask      = well_ids == well
        n_in_well = int(mask.sum())
        sub_idx   = (np.arange(n_in_well) if max_per_well is None or n_in_well <= max_per_well
                    else rng.choice(n_in_well, size=max_per_well, replace=False))
        ax.plot(t, X[mask][sub_idx].T, color=label_color[label], lw=0.4, alpha=alpha,
                zorder=1, rasterized=True)

    seen = set()
    for well, label in zip(unique_wells, well_labels):  # foreground: per-well mean, same colour, full opacity
        if(with_nc==False and label=="NC-ALL"):
                continue
        if(with_pc==False and label=="PC"):
                continue
        if(nc_only==True and label!="NC-ALL"):
                continue
        mean = X[well_ids == well].mean(axis=0)
        ax.plot(t, mean, color=label_color[label], lw=2.2, alpha=1.0, zorder=2,
                label=None if label in seen else label)
        seen.add(label)

    if caption is not None:
        ax.text(0.5, -0.15, caption, transform=ax.transAxes, ha="center", va="top",
                fontsize=14, fontweight="bold")
    else:
        ax.set_title(f"{X_name} — all curves by label + per-well mean\n{exp_path.name}",
                    fontsize=11, fontweight="bold")
    ax.set_xlabel("Time (s)" if x_time is not None else "time")
    # ax.set_ylabel(X_name)
    if y_limit is not None:
        bottom, top = y_limit
        ax.set_ylim(bottom, top)
    ax.tick_params(labelsize=10)
    ax.grid(True, color="grey", alpha=0.15, linestyle=":")
    if show_legend:
        ax.legend(title=legend_title, fontsize=11, framealpha=0.85, loc="upper right")
    if fig is not None:
        plt.tight_layout()
    return fig


def short_name(folder_name):
    m = re.search(r'DDM_0(\d)', folder_name)
    return f'Chip 0{m.group(1)}' if m else folder_name.split('_U_', 1)[1]


def plot_raw_vs_attn(X_AC, X_attn, well_ids, exp_path, curve_type, attn_model, with_nc=False, with_pc=False,
                     x_time=None):
    """1x2 figure: left = raw input curves, right = attn-recon curves, same per-well-mean
    style as plot_all_curves_by_label. Figure title = short_name(exp_path.name); each
    panel captioned (a)/(b) below instead of titled above. x_time (T,), if given, is used
    as the x-axis (real experiment seconds) instead of a bare sample index."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), facecolor="white", sharey=True)

    plot_all_curves_by_label(X_AC, well_ids, exp_path, X_name=f"Original curve ({curve_type})",
                             with_nc=with_nc, with_pc=with_pc, ax=axes[0], caption="(a) Input Curves", x_time=x_time,
                             show_legend=False)
    plot_all_curves_by_label(X_attn, well_ids, exp_path, X_name=f"Attn recon ({attn_model})",
                             with_nc=with_nc, with_pc=with_pc, ax=axes[1],
                             caption="(b) Spatial Attn Pooling Layer\nReshaped Curves", x_time=x_time,
                             show_legend=False)

    fig.suptitle(short_name(exp_path.name), fontsize=14, fontweight="bold", y=1.02)
    fig.tight_layout()
    fig.subplots_adjust(right=0.85, bottom=0.24)
    handles, labels = axes[1].get_legend_handles_labels()
    fig.legend(handles, labels, title="target", fontsize=11, framealpha=0.85,
              loc="center left", bbox_to_anchor=(0.87, 0.55))
    return fig


In [ ]:
%matplotlib inline

_suffix    = {0: "", 1: "_supcon", 3: "_supcon3"}[SUPCON]
ATTN_MODEL = f"cnn_gru_dual_attn_recon{_suffix}"
print(f"Attn model   : {ATTN_MODEL}")

dataset_names = config.CROSS_DATASET_GROUPS[GROUP_NAME]
print(f"Group '{GROUP_NAME}' -> {len(dataset_names)} chips")


In [ ]:
# Per chip in final_6_new: raw vs attn-recon curves, well comparison, neighbour grid
for dataset_name in dataset_names:
    exp_path_i = Path(EXP_FOLDER, dataset_name)
    X_AC_i, coords_i, well_ids_i, Y_well_i, X_time_i, exp_path_i = load_experiment_data(
        None, None, curve_type=CURVE_TYPE, outlier_filter=OUTLIER_FILTER, exp_path=exp_path_i,
    )
    neighbor_stack_i = build_neighbor_curve_stack(X_AC_i, coords_i, well_ids_i, k=K_NEIGHBORS)

    try:
        _, X_attn_i, attn_weights_i = build_recon_attn(
            X_AC_i, coords_i, well_ids_i,
            model_path=_attn_model_path(exp_path_i, ATTN_MODEL, OUTLIER_FILTER, CURVE_TYPE),
            k=K_NEIGHBORS,
        )
    except FileNotFoundError as e:
        print(f"[!] {e}")
        X_attn_i, attn_weights_i = None, None

    fig = plot_raw_vs_attn(X_AC_i, X_attn_i, well_ids_i, exp_path_i, CURVE_TYPE, ATTN_MODEL,
                           with_nc=False, with_pc=False, x_time=X_time_i)
    plt.show()
    plt.close(fig)

    fig = plot_well_comparison(X_AC_i, X_attn_i, well_ids_i, exp_path_i, SUPCON, ATTN_MODEL)
    plt.show()
    plt.close(fig)

    rng     = np.random.default_rng(42)
    indices = rng.choice(len(X_AC_i), size=min(N_SAMPLE, len(X_AC_i)), replace=False)
    indices.sort()
    fig = plot_recon_grid_combined(neighbor_stack_i, X_attn_i, indices, ATTN_MODEL, exp_path_i,
                                   attn_weights=attn_weights_i)
    plt.show()
    plt.close(fig)
